# Week 7 — Simple classroom walkthrough

**COM7024 · Lesson 7 · SciPy**  
A short live demonstration of the four main SciPy ideas. Suggested time: 20–25 minutes.

**How to use:** Run the cells from top to bottom. Every example works with the supplied full dataset.

## Step 0 — Find the data folder

This short setup supports your GitHub layout: `app/L7/` for notebooks and `data/` at the repository root.

In [ ]:
from pathlib import Path

# The data folder is at the root of your GitHub repository.
DATA = Path("data")                 # If Jupyter starts at the repository root
if not DATA.exists():
    DATA = Path("../../data")       # If Jupyter starts inside app/L7
if not DATA.exists():
    DATA = Path("../data")          # If notebooks are in a separate folder

print("Data folder:", DATA.resolve())

## 1. Look at the real data

Read the complete weekly Mauna Loa CO₂ dataset. `ppm` means parts per million.

In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "MaunaLoa_CO2_Full.csv", parse_dates=["date"])
print("Rows:", len(df))
print("Missing CO2 readings:", df["co2_ppm"].isna().sum())
display(df.head())

## 2. Draw the trend

One line of pandas plots the measurements; gaps are left as gaps.

In [ ]:
import matplotlib.pyplot as plt

df.plot(x="date", y="co2_ppm", figsize=(10, 4), title="Weekly CO2 readings")
plt.ylabel("CO2 (ppm)")
plt.show()

## 3. Integrate a simple function

`quad` returns two values: the integral and an estimate of its numerical error.

In [ ]:
from scipy.integrate import quad

def f(x):
    return x**2

area, error = quad(f, 0, 2)
print("Area:", round(area, 4))
print("Estimated error:", error)

## 4. Integrate the measured data

For a dataset, use `trapezoid`. The result here has units **ppm × years**.

In [ ]:
from scipy.integrate import trapezoid

filled = df["co2_ppm"].interpolate()
years = (df["date"] - df["date"].min()).dt.days / 365.25
area = trapezoid(filled - 315, x=years)
print("Time-integrated CO2 above 315 ppm:", round(area, 2), "ppm-years")

## 5. Fill missing values

Interpolation estimates the readings between observed dates. It does not create new measurements.

In [ ]:
df["filled"] = df["co2_ppm"].interpolate()
print("Missing before:", df["co2_ppm"].isna().sum())
print("Missing after:", df["filled"].isna().sum())

## 6. Group years by similar averages

`kmeans2` puts annual average concentrations into two groups. The group numbers have no ranking.

In [ ]:
from scipy.cluster.vq import kmeans2

annual = df.groupby(df["date"].dt.year)["filled"].mean()
centres, groups = kmeans2(annual.to_numpy().reshape(-1, 1), 2,
                          minit="++", seed=7)
print("First 10 group labels:", groups[:10])

## 7. Solve a small system of equations

Two equations, two unknown prices. `solve` finds both prices.

In [ ]:
import numpy as np
from scipy.linalg import solve

A = np.array([[2, 1], [1, 2]])
b = np.array([8, 7])
print("Coffee and tea prices:", solve(A, b))

## 8. Smooth a picture

SciPy can also process a picture because an image is a grid of numbers.

In [ ]:
from PIL import Image
from scipy.ndimage import gaussian_filter

picture = np.array(Image.open(DATA / "Camera_Practice_Image.png").convert("L"))
smooth = gaussian_filter(picture, sigma=3)
plt.imshow(smooth, cmap="gray")
plt.title("Smoothed image")
plt.axis("off")
plt.show()

## Your turn — Quick challenge

Choose one of the four techniques and explain in one sentence **what question it answers**. Then change `sigma=3` to `sigma=1` in the final cell and observe the difference.

### Before you finish

**Discuss:** Integration accumulates values; interpolation estimates gaps; clustering groups similar items; image filters change pixel values.